
🌶️ 01 — Flask Basics: Diner Menu-Card Style Web! 🍛

**Topic 13.03 | Other_Libraries — Part 3 of 8**

🌍 **Hook:** The diner-owner's "SHOUT-IT" system: behind a counter, the server
shouts any item and the plate is instantly ready! Flask is that kind of LIGHTWEIGHT
web-framework — small, tidy, straight-forward: the moment you write a route, the
website's pages are ready! FastAPI is the fine-dine POS (JSON-sequence); Flask is
the diner (HTML-plates! 🍛).

📖 **Definitions:**
- `@app.route("/")` = writing the page's address (an item on the menu-card!)
- **test_client()** = delivery-rehearsal — try the page without starting a server! 🎭
- **jsonify** = dict → a proper-JSON plate (headers-logs-right!)
- `<int:pid>` = the URL's int-slot (42 OK, "ukta" → auto-404!)

🔍 **WHY learn it?** Rapid small sites/demo-pages, internal tools, webhook-receivers
— a website standing in 10 lines!


## 🐣 Level 0 — First Route: Hello-Page! 👋


In [1]:
from flask import Flask, jsonify, request

app = Flask("dhaba-app 🍛")

@app.route("/")
def home():
    return "<h1>🍛 Flask-Dhaba: Aao Ji Aao!</h1>"

client = app.test_client()
r = client.get("/")
print("status:", r.status_code)
print("html  :", r.data.decode())
# Expected OUTPUT:
# status: 200
# html  : <h1>🍛 Flask-Dhaba: Aao Ji Aao!</h1>

status: 200
html  : <h1>🍛 Flask-Dhaba: Aao Ji Aao!</h1>



## 🐥 Level 1 — URL-Slots, Query & JSON 🧾

- `<int:pid>` — an int-only slot (pass a string → 404 auto!)
- `request.args` — the `?limit=2`-style extras
- `jsonify(...)` — a JSON-response with clean headers


In [2]:
# ⚠️ WALEHA-UPDATE: asali-dev me naya route add = server RESTART karna!
# notebook me usi ka roop = NAYA Flask app (pehle wala freeze ho chuka — ek request ke baad Flask route-bandh kar deta!)
app2 = Flask("dhaba-v2 🍛")
DISHES = [
    {"id": 1, "name": "Dal Fry",   "price": 90},
    {"id": 2, "name": "Paneer",    "price": 160},
    {"id": 3, "name": "Naan",      "price": 25},
    {"id": 4, "name": "Chai",      "price": 12},
]

@app2.route("/menu")
def menu():
    limit = int(request.args.get("limit", "4"))     # query-magic!
    return jsonify({"count": min(limit, len(DISHES)), "dishes": DISHES[:limit]})

@app2.route("/dish/<int:pid>")
def dish(pid):
    for d in DISHES:
        if d["id"] == pid:
            return jsonify(d)
    return jsonify({"error": f"dish-{pid} menu me nahi"}), 404    # tuple → (body, status)!

client2 = app2.test_client()
print(client2.get("/menu?limit=2").get_json()["count"], "dishes aaye limit=2 pe!")
print("dish/3 :", client2.get("/dish/3").get_json())
print("dish/9 :", client2.get("/dish/9").status_code)
print("dish/xy:", client2.get("/dish/xy").status_code, "(int-slot ne auto-reject! 👮)")
# Expected OUTPUT:
# {'count': 2, 'dishes': [...2 items...]}
# dish/3 : {'id': 3, 'name': 'Naan', 'price': 25}
# dish/9 : 404
# dish/xy: 404 (int-slot ne auto-reject! 👮)

2 dishes aaye limit=2 pe!
dish/3 : {'id': 3, 'name': 'Naan', 'price': 25}
dish/9 : 404
dish/xy: 404 (int-slot ne auto-reject! 👮)



## 🐔 Level 2 — POST-Order, Redirect & Error-Page 🧍

- `methods=["POST"]` + `request.form` = to pick up the order-form
- `redirect(url_for("..."))` = "send it to the order-hall!"
- `@app.errorhandler(404)` = your own custom error-page (JSON too!)


In [3]:
from flask import url_for, redirect

app3 = Flask("dhaba-v3 🍛")        # phir se "restart" (naya app) — routes pehle, order baad!
ORDERS = []

@app3.route("/order", methods=["POST"])
def take_order():
    dish_id = int(request.form["dish_id"])
    qty     = int(request.form.get("qty", 1))
    for d in DISHES:
        if d["id"] == dish_id:
            bill = d["price"] * qty
            ORDERS.append({"dish": d["name"], "qty": qty, "bill": bill})
            return redirect(url_for("summary"))          # order-hall!
    return jsonify({"error": "dish bekaar!"}), 400

@app3.route("/summary")
def summary():
    total = sum(o["bill"] for o in ORDERS)
    return jsonify({"orders": ORDERS, "total": total})

@app3.errorhandler(404)
def page_missing(e):
    return "<h2>😶 dhaba ye page nahi pakta!</h2>", 404

c3 = app3.test_client()
c3.post("/order", data={"dish_id": "2", "qty": "2"})   # form-data!
c3.post("/order", data={"dish_id": "4", "qty": "3"})
print("bad-order:", c3.post("/order", data={"dish_id": "99"}).status_code)
print("summary :", c3.get("/summary").get_json())
print("404-page:", c3.get("/kuch-bhi").data.decode()[:42])
# Expected OUTPUT:
# bad-order: 400
# summary : {'orders': [{'dish': 'Paneer', 'qty': 2, 'bill': 320}, {'dish': 'Chai', 'qty': 3, 'bill': 36}], 'total': 356}
# 404-page: <h2>😶 dhaba ye page nahi pakta!</h2>

bad-order: 400
summary : {'orders': [{'bill': 320, 'dish': 'Paneer', 'qty': 2}, {'bill': 36, 'dish': 'Chai', 'qty': 3}], 'total': 356}
404-page: <h2>😶 dhaba ye page nahi pakta!</h2>



---
## ⚠️ 5 Classic Mistakes (Flask)
| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | using the form-value directly as int (everything comes as str!) | `int(request.form[...])` |
| 2 | returning the dict directly and fixing the status on another line (confusing) | `jsonify(...)`, 404 tuple-style |
| 3 | `app.run()` inside the notebook (blocking!) | test_client rehearsal 🎭 |
| 4 | hard-coding the URL ("/summary") | `url_for("summary")` refactor-safe |
| 5 | assuming POST is a default method (only GET!) | write `methods=["POST"]` |

## 📌 Recap — 30 Second
- `@app.route + return` = page · `<int:slot>` auto-police 👮
- query = `request.args` · form = `request.form` · JSON = `jsonify`
- `(body, 404)` status-tuple · `redirect(url_for(...))` · custom errorhandler
- test_client = offline-rehearsal — for a real run use `flask run`! 🎭→🚀

**Next:** `02_routes_templates.ipynb` — Jinja's magic: putting python on HTML
pages (loops/ifs/inheritance/i18n-clean products catalog!) ✨
